# Module 03 — Lesson 08
# Matrices and Matrix Multiplication

---

> Lesson 07 showed you that a single row of data — one student's record — is a vector. What about the WHOLE dataset, every student at once? Stack those row-vectors together, and you get a **matrix**. This is exactly what a pandas DataFrame or a NumPy array is, underneath all the convenient syntax: a grid of numbers.

Matrices are everywhere once you know to look:
- A spreadsheet or dataset — rows are records, columns are features
- A grayscale image — each pixel's brightness is one entry in a matrix (you'll build on this directly in Module 04's project)
- A neural network layer — transforms one matrix of numbers into another via matrix multiplication

By the end of this lesson, you'll implement matrix multiplication from scratch — the single operation that powers linear regression predictions, neural network layers, and image transformations.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Define a matrix as a 2D grid of numbers and represent one as a list of row-vectors in Python
- Represent a dataset as a matrix, and explain what "shape" (m × n) means
- Perform matrix addition, subtraction, and scalar multiplication from scratch
- Compute the transpose of a matrix and explain when it's useful
- Multiply two matrices from scratch, and correctly apply the dimension compatibility rule
- Explain why matrix multiplication is NOT element-wise, and why it's not commutative

---
## 1. What Is a Matrix?

A **matrix** is a 2D grid of numbers, arranged in rows and columns. In Python, we represent one as a list of row-vectors — a list of lists, where each inner list is one row (exactly the vectors from Lesson 07).

In [ ]:
# Three students, each a vector [hours_studied, exam_score, attendance_pct]
# Stack them into rows -> a matrix
students_matrix = [
    [8, 85, 92],   # Priya
    [3, 55, 70],   # Rohan
    [7, 80, 88],   # Meera
]

print("students_matrix (a list of row-vectors):")
for row in students_matrix:
    print(f"  {row}")

print()
print(f"Row 0 (Priya's vector)        : {students_matrix[0]}")
print(f"Row 0, Column 1 (Priya's score): {students_matrix[0][1]}")
print()
print("A matrix is just a stack of the vectors from Lesson 07 — nothing new")
print("mathematically, just a new way to organize many vectors at once.")

---
## 2. Representing a Dataset as a Matrix

| | Column 0 (hours) | Column 1 (score) | Column 2 (attendance) |
|---|---|---|---|
| Row 0 (Priya) | 8 | 85 | 92 |
| Row 1 (Rohan) | 3 | 55 | 70 |
| Row 2 (Meera) | 7 | 80 | 88 |

This is precisely what happens when you load a CSV into pandas or NumPy: **rows become records, columns become features**, and the whole table is one matrix. Module 04 (NumPy) and Module 05 (pandas) build directly on this idea with much more convenient syntax — but the underlying structure is exactly what you're building here by hand.

In [ ]:
def print_matrix(M, label=""):
    '''Pretty-print a matrix with aligned columns.'''
    if label:
        print(f"  {label}")
    for row in M:
        print("   [" + "  ".join(f"{x:>6}" for x in row) + "]")
    print()


print_matrix(students_matrix, label="students_matrix (3 students x 3 features)")
print("Reading a single feature (attendance, column 2) across ALL students:")
attendance_column = [row[2] for row in students_matrix]
print(f"  {attendance_column}")

---
## 3. Matrix Shape and Notation

A matrix with **m** rows and **n** columns has **shape (m, n)**, often written $m \times n$. `students_matrix` above is $3 \times 3$. A matrix doesn't have to be square — a matrix of 100 students with 5 features each would be $100 \times 5$.

In [ ]:
def shape(M):
    '''Return (num_rows, num_cols) for a matrix M.'''
    return (len(M), len(M[0]) if M else 0)


square = [[1, 2], [3, 4]]
rectangular = [[1, 2, 3], [4, 5, 6]]

print(f"square       = {square}       -> shape {shape(square)}")
print(f"rectangular  = {rectangular} -> shape {shape(rectangular)}")
print(f"students_matrix           -> shape {shape(students_matrix)}")

---
## 4. Matrix Addition, Subtraction, and Scalar Multiplication

Just like vectors, these operations work **component-wise** — and require both matrices to have the **exact same shape**.

In [ ]:
def matrix_add(A, B):
    '''Add two matrices element-wise. Must have identical shapes.'''
    if shape(A) != shape(B):
        raise ValueError(f"Cannot add matrices of shape {shape(A)} and {shape(B)}")
    return [[a + b for a, b in zip(row_a, row_b)] for row_a, row_b in zip(A, B)]

def matrix_subtract(A, B):
    '''Subtract matrix B from matrix A, element-wise. Must have identical shapes.'''
    if shape(A) != shape(B):
        raise ValueError(f"Cannot subtract matrices of shape {shape(A)} and {shape(B)}")
    return [[a - b for a, b in zip(row_a, row_b)] for row_a, row_b in zip(A, B)]

def scalar_multiply_matrix(k, M):
    '''Multiply every entry of matrix M by scalar k.'''
    return [[k * x for x in row] for row in M]


# Two exam attempts for the same 3 students (rows), 2 sections (columns)
attempt_1 = [[70, 65], [80, 75], [60, 55]]
attempt_2 = [[75, 70], [85, 80], [65, 60]]

combined = matrix_add(attempt_1, attempt_2)
bonus_5pct = scalar_multiply_matrix(1.05, combined)

print_matrix(attempt_1, "attempt_1")
print_matrix(attempt_2, "attempt_2")
print_matrix(combined, "attempt_1 + attempt_2")
print_matrix(bonus_5pct, "(attempt_1 + attempt_2) * 1.05  (a 5% bonus applied)")

---
## 5. The Transpose

The **transpose** of a matrix flips it across its diagonal — rows become columns, and columns become rows. A $(m, n)$ matrix transposes into an $(n, m)$ matrix.

In [ ]:
def transpose(M):
    '''Flip a matrix's rows and columns.'''
    rows, cols = shape(M)
    return [[M[r][c] for r in range(rows)] for c in range(cols)]


print_matrix(students_matrix, "students_matrix (shape " + str(shape(students_matrix)) + ")")
students_T = transpose(students_matrix)
print_matrix(students_T, "transpose(students_matrix) (shape " + str(shape(students_T)) + ")")

print("Notice: each ROW of the transpose is now a FEATURE column from the")
print("original — [8, 3, 7] is every student's 'hours_studied' value.")
print("This is handy when you want to compute per-feature statistics")
print("(like the mean of each column) using the row-based tools from Lesson 02.")
print()
print(f"Transpose of the transpose returns the original: "
      f"{transpose(students_T) == students_matrix}")

---
## 6. Matrix Multiplication — The Rule

Matrix multiplication is **not** element-wise. To multiply matrix A ($m \times n$) by matrix B ($n \times p$):

- The **inner dimensions must match**: A's number of columns (n) must equal B's number of rows (n)
- The result has shape ($m \times p$) — outer dimensions
- Each entry of the result is the **dot product of a row of A with a column of B** (a preview of Lesson 09!)

$$C_{ij} = \sum_{k=1}^{n} A_{ik} \, B_{kj}$$

In [ ]:
def matrix_multiply(A, B):
    '''Multiply matrix A by matrix B. Requires cols(A) == rows(B).'''
    rows_a, cols_a = shape(A)
    rows_b, cols_b = shape(B)
    if cols_a != rows_b:
        raise ValueError(
            f"Cannot multiply {shape(A)} by {shape(B)}: "
            f"inner dimensions must match ({cols_a} != {rows_b})"
        )

    result = []
    for i in range(rows_a):
        row = []
        for j in range(cols_b):
            value = sum(A[i][k] * B[k][j] for k in range(cols_a))
            row.append(value)
        result.append(row)
    return result


A = [[1, 2, 3], [4, 5, 6]]      # shape (2, 3)
B = [[7, 8], [9, 10], [11, 12]]  # shape (3, 2)

C = matrix_multiply(A, B)

print_matrix(A, f"A, shape {shape(A)}")
print_matrix(B, f"B, shape {shape(B)}")
print_matrix(C, f"A @ B, shape {shape(C)}")

# Verify one entry by hand: C[0][0] = row 0 of A dot column 0 of B
manual = A[0][0]*B[0][0] + A[0][1]*B[1][0] + A[0][2]*B[2][0]
print(f"Hand-check C[0][0] = 1*7 + 2*9 + 3*11 = {manual}  (matches C[0][0]={C[0][0]})")

---
## 7. Matrix Multiplication Is NOT Element-wise

In [ ]:
def elementwise_multiply(A, B):
    '''Multiply two SAME-SHAPE matrices entry-by-entry (NOT true matrix multiplication).'''
    if shape(A) != shape(B):
        raise ValueError(f"Cannot elementwise-multiply shapes {shape(A)} and {shape(B)}")
    return [[a * b for a, b in zip(row_a, row_b)] for row_a, row_b in zip(A, B)]


A = [[1, 2], [3, 4]]
B = [[5, 6], [7, 8]]

elementwise_result = elementwise_multiply(A, B)
matmul_result = matrix_multiply(A, B)

print_matrix(A, "A")
print_matrix(B, "B")
print_matrix(elementwise_result, "elementwise_multiply(A, B) -- just A[i][j]*B[i][j]")
print_matrix(matmul_result, "matrix_multiply(A, B) -- row dot column")
print("Same input matrices, COMPLETELY different results and different meaning.")

---
## 8. The Identity Matrix

The **identity matrix** I is the matrix equivalent of the number 1 — a square matrix with 1s on the diagonal and 0s everywhere else. Multiplying any matrix by the (correctly-sized) identity matrix leaves it unchanged.

In [ ]:
def identity_matrix(n):
    '''An n x n identity matrix: 1s on the diagonal, 0s elsewhere.'''
    return [[1 if i == j else 0 for j in range(n)] for i in range(n)]


I3 = identity_matrix(3)
print_matrix(I3, "identity_matrix(3)")

result = matrix_multiply(students_matrix, I3)
print(f"students_matrix @ I3 == students_matrix : {result == students_matrix}")
print("Just like multiplying a number by 1 leaves it unchanged.")

---
## 9. Applied: Weighted Scores via Matrix Multiplication

A very common real-world pattern: combine several scores into one final score using fixed weights. This is a **linear combination** — and computing it for many students at once is exactly matrix multiplication. (This is also, not coincidentally, the exact computation a linear regression model performs to make a prediction — you'll meet that formally in Module 11.)

In [ ]:
# 4 students, 3 component scores each: [quiz, midterm, final_exam]
students_scores = [
    [80, 85, 90],   # Aisha
    [60, 70, 75],   # Vikram
    [95, 90, 88],   # Sara
    [70, 65, 72],   # Dev
]

# Weights: quiz 20%, midterm 20%, final exam 60% -- as a (3 x 1) column matrix
weights = [[0.20], [0.20], [0.60]]

print_matrix(students_scores, f"students_scores, shape {shape(students_scores)}")
print_matrix(weights, f"weights (column matrix), shape {shape(weights)}")

final_grades = matrix_multiply(students_scores, weights)
print_matrix(final_grades, f"students_scores @ weights = final grades, shape {shape(final_grades)}")

names = ["Aisha", "Vikram", "Sara", "Dev"]
for name, row in zip(names, final_grades):
    print(f"  {name:<8}: {row[0]:.1f}")

---
## 10. Sanity-Check: Verifying a Known Matrix Property

Since this module works with pure Python (no NumPy yet — that's Module 04), we can't cross-check against a library. Instead, we verify our functions against a **known mathematical property**: the transpose of a product equals the product of the transposes, in reverse order.

$$(AB)^T = B^T A^T$$

In [ ]:
A = [[1, 2, 3], [4, 5, 6]]       # shape (2, 3)
B = [[7, 8], [9, 10], [11, 12]]   # shape (3, 2)

left_side = transpose(matrix_multiply(A, B))
right_side = matrix_multiply(transpose(B), transpose(A))

print_matrix(left_side, "(A @ B)^T")
print_matrix(right_side, "B^T @ A^T")
print(f"Property (A@B)^T == B^T@A^T holds: {left_side == right_side}")
print()
print("This property holding confirms our matrix_multiply() and transpose()")
print("implementations agree with the standard mathematical definition — the")
print("same functions NumPy uses internally (Module 04 does this in one line).")

---
## ⚠️ Common Mistakes

In [ ]:
# -- Mistake 1: Adding/subtracting matrices of mismatched shapes --------------

A = [[1, 2], [3, 4]]
B = [[1, 2, 3], [4, 5, 6]]

print("Mistake 1 — Adding matrices of different shapes:")
try:
    matrix_add(A, B)
except ValueError as e:
    print(f"  Caught error: {e}")
print()
print("  Just like vectors, matrix addition/subtraction is element-wise and")
print("  needs an EXACT shape match on both dimensions, not just row count.")

In [ ]:
# -- Mistake 2: Confusing element-wise multiplication with real matrix multiplication

A = [[1, 2], [3, 4]]
B = [[5, 6], [7, 8]]

wrong_thinking = elementwise_multiply(A, B)   # NOT what "matrix multiplication" means
correct = matrix_multiply(A, B)

print("Mistake 2 — Element-wise multiply is NOT matrix multiplication:")
print(f"  elementwise_multiply(A, B) = {wrong_thinking}")
print(f"  matrix_multiply(A, B)      = {correct}")
print()
print("  When people say 'multiply these matrices,' they almost always mean")
print("  TRUE matrix multiplication (row dot column) unless explicitly told")
print("  otherwise (NumPy calls element-wise multiply the '*' operator, and")
print("  true matrix multiplication '@' — they are DIFFERENT operators).")

In [ ]:
# -- Mistake 3: Getting the dimension compatibility rule backwards ------------

A = [[1, 2, 3]]        # shape (1, 3)
B = [[1, 2, 3]]        # shape (1, 3) -- same shape, but WRONG for multiplication

print("Mistake 3 — Forgetting the inner-dimension rule:")
try:
    matrix_multiply(A, B)
except ValueError as e:
    print(f"  Caught error: {e}")
print()
print("  A's COLUMN count must match B's ROW count -- not just 'similar shapes.'")
print(f"  Fix: multiply A (shape {shape(A)}) by transpose(B) (shape {shape(transpose(B))}) instead.")
fixed = matrix_multiply(A, transpose(B))
print(f"  A @ transpose(B) = {fixed}")

In [ ]:
# -- Mistake 4: Assuming matrix multiplication is commutative -----------------

A = [[1, 2], [3, 4]]
B = [[2, 0], [1, 2]]

AB = matrix_multiply(A, B)
BA = matrix_multiply(B, A)

print("Mistake 4 — Assuming A @ B == B @ A (it usually doesn't!):")
print_matrix(AB, "A @ B")
print_matrix(BA, "B @ A")
print(f"A @ B == B @ A : {AB == BA}")
print()
print("  Unlike multiplying plain numbers, ORDER MATTERS for matrix")
print("  multiplication. Always keep track of which matrix comes first —")
print("  swapping the order can produce a completely different (or even")
print("  differently-shaped) result.")

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Build a Matrix and Check Its Shape

Build a matrix from these three product records `[price, rating, stock]` and print its shape using `shape()`.

```python
product_a = [500, 4.2, 80]
product_b = [520, 4.5, 75]
product_c = [150, 3.8, 95]
```

In [ ]:
product_a = [500, 4.2, 80]
product_b = [520, 4.5, 75]
product_c = [150, 3.8, 95]
# Your code here

### Exercise 2 — Addition and Transpose

Given `rainfall_2023` and `rainfall_2024` (each a 3x2 matrix of monthly rainfall in mm for 3 cities across 2 months), compute the total rainfall matrix, then transpose it so months become rows instead of cities.

```python
rainfall_2023 = [[120, 90], [80, 60], [150, 110]]
rainfall_2024 = [[100, 95], [85, 70], [140, 130]]
```

In [ ]:
rainfall_2023 = [[120, 90], [80, 60], [150, 110]]
rainfall_2024 = [[100, 95], [85, 70], [140, 130]]
# Your code here

### Exercise 3 — Matrix Multiplication By Hand and By Code

Given `A = [[2, 0], [1, 3]]` and `B = [[4, 1], [2, 2]]`, compute `A @ B` using `matrix_multiply()`. Then verify the entry `C[1][0]` by hand-calculating the dot product of row 1 of A and column 0 of B.

In [ ]:
A = [[2, 0], [1, 3]]
B = [[4, 1], [2, 2]]
# Your code here

### Exercise 4 — Weighted Course Grade Calculator

Five students each have 4 component scores `[assignment, quiz, midterm, final]`. Using matrix multiplication, compute each student's final grade with weights `[0.10, 0.15, 0.25, 0.50]`.

```python
scores = [
    [85, 78, 82, 90],
    [70, 65, 60, 72],
    [95, 92, 88, 91],
    [55, 60, 58, 65],
    [80, 82, 79, 85],
]
```

In [ ]:
scores = [
    [85, 78, 82, 90],
    [70, 65, 60, 72],
    [95, 92, 88, 91],
    [55, 60, 58, 65],
    [80, 82, 79, 85],
]
# Your code here

### Exercise 5 — (Challenge) Verify Non-Commutativity Systematically

Write `check_commutative(A, B)` that returns `True` if `matrix_multiply(A, B) == matrix_multiply(B, A)` (both must be defined and equal), and `False` otherwise (catch shape errors and treat them as `False`). Test it on several pairs of square matrices, including the identity matrix (which SHOULD commute with anything).

In [ ]:
def check_commutative(A, B):
    # Your code here
    pass

test_pairs = [
    ([[1, 2], [3, 4]], identity_matrix(2)),
    ([[1, 2], [3, 4]], [[2, 0], [1, 2]]),
]
for A, B in test_pairs:
    print(check_commutative(A, B))

---
## 📌 Key Takeaways

- **A matrix is a stack of vectors — rows are records, columns are features — and this IS what a pandas DataFrame or NumPy array is underneath.** Addition, subtraction, and scalar multiplication all work element-wise, just like vectors, and require matching shapes.

- **Matrix multiplication is fundamentally different from element-wise multiplication: it's built from repeated dot products (row · column), requires the inner dimensions to match, and is NOT commutative.** `A @ B` almost never equals `B @ A` — always track which matrix comes first.

- **"Weighted combination of columns" — like computing a final grade from component scores — is exactly what matrix multiplication computes, and it's the same operation a linear regression model uses to turn features into a prediction.** You'll see this pattern again and again throughout the ML modules.

---
## What's Next?

**Lesson 09 — Dot Product and Its Applications**
Every entry of every matrix multiplication you just computed was secretly a **dot product** — a row of one matrix combined with a column of another. Next lesson pulls that operation out on its own, and explores what it means geometrically and why it's called "the engine inside linear regression and neural networks."